In [1]:
import pandas as pd

folder = r"D:\retailpulse\datarawsynthetic\retail_contaminated_dataset"

customers = pd.read_csv(folder + r"\customer_master.csv")   # who the customers are
products  = pd.read_csv(folder + r"\sku_master.csv")        # what products we sell
stores    = pd.read_csv(folder + r"\store_master.csv")      # where we sell

# both tables have a "city" column, so give them clear names
customers = customers.rename(columns={"city": "customer_city"})
stores    = stores.rename(columns={"city": "store_city"})
# the price list is not the same as the price on the sale, so rename it
products  = products.rename(columns={"unit_price": "list_price"})

In [2]:
sales = pd.read_csv(
    folder + r"\sales_transactions.csv",
    parse_dates=["date"],      # read the date column as real dates
    # nrows=2_000_000,         # remove the # if your laptop struggles: it loads only 2 million rows
)
print("Rows loaded:", len(sales))

Rows loaded: 9945511


In [3]:
# 1. remove exact duplicate rows
before = len(sales)
sales = sales.drop_duplicates()
print("Duplicate rows removed:", before - len(sales))

# 2. look for missing values
print(sales.isna().sum())

Duplicate rows removed: 12897
date                  0
receipt_id            0
store_id              0
sku_id                0
customer_id           0
quantity              0
unit_price            0
total_value           0
channel               0
discount_pct          0
promo_id        7845164
dtype: int64


In [4]:
data = sales.merge(products, on="sku_id", how="left")                         # add product details
data = data.merge(customers, left_on="customer_id", right_on="cust_id", how="left")  # add customer details
data = data.merge(stores, on="store_id", how="left")                          # add store details

data = data.drop(columns=["cust_id", "list_price"])   # we don't need these two
print(data.shape)

(9932614, 26)


In [5]:
# --- money columns ---
data["gross_value"]      = data["quantity"] * data["unit_price"]        # price before discount
data["discount_amount"]  = data["gross_value"] - data["total_value"]    # money given as discount
data["estimated_profit"] = data["total_value"] * 0.30                   # mentor's rule: 30% margin
data["estimated_cost"]   = data["total_value"] - data["estimated_profit"]
data["real_profit"]      = data["total_value"] - data["quantity"] * data["cost_price"]  # bonus: uses real cost

# --- time columns ---
data["year"]       = data["date"].dt.year
data["quarter"]    = "Q" + data["date"].dt.quarter.astype(str)
data["month"]      = data["date"].dt.month
data["year_month"] = data["date"].dt.strftime("%Y-%m")
data["weekday"]    = data["date"].dt.day_name()

season = {12: "Winter", 1: "Winter", 2: "Winter", 3: "Spring", 4: "Spring", 5: "Spring",
          6: "Summer", 7: "Summer", 8: "Summer", 9: "Autumn", 10: "Autumn", 11: "Autumn"}
data["season"] = data["month"].map(season)

# --- business columns ---
data["has_promo"]  = data["promo_id"].notna()                       # True if the sale had a promotion
data["price_band"] = pd.qcut(data["unit_price"], 3, labels=["Low", "Mid", "High"])
data["age_group"]  = pd.cut(data["age"], [17, 25, 35, 50, 80],
                            labels=["18-25", "26-35", "36-50", "51+"])

In [6]:
print(data.shape)
print(data[["date", "quantity", "total_value", "estimated_profit", "real_profit",
            "season", "has_promo", "price_band", "age_group"]].head())

data.to_parquet(r"D:\retailpulse\retailpulse_sales_clean.parquet", index=False)

(9932614, 40)
        date  quantity  total_value  estimated_profit  real_profit  season  \
0 2025-04-02         1      2379.11           713.733       751.20  Spring   
1 2025-04-02         3      1006.65           301.995       278.28  Spring   
2 2022-04-24         1       820.53           246.159       167.07  Spring   
3 2022-04-24         2       176.64            52.992        77.32  Spring   
4 2024-09-22         1      1660.93           498.279       369.75  Autumn   

   has_promo price_band age_group  
0      False       High     36-50  
1      False        Mid     36-50  
2      False       High       51+  
3      False        Low       51+  
4      False       High       51+  
